# Hessian-vector products, forward over reverse

Second-order methods and curvature diagnostics need products $Hv$ with the Hessian $H = \nabla^2 f(w)$. Forming $H$ costs $O(D^2)$ memory, but $Hv$ is the directional derivative of the gradient, which JAX computes for about the cost of two gradients.

Use the regularised logistic loss with labels $y_i \in \{-1, +1\}$:

$$f(w) = \frac{1}{N} \sum_{i=1}^{N} \mathrm{softplus}(-y_i\, x_i^\top w) + \frac{\lambda}{2} \lVert w \rVert^2, \qquad \mathrm{softplus}(t) = \log(1 + e^t)$$

Write `grad_and_hvps(w, X, y, lam, V)`:

- `w`: `(D,)`; `X`: `(N, D)`; `y`: `(N,)` of $\pm 1$; `lam`: float.
- `V`: `(k, D)`: $k$ directions.

Return `(g, HV)`: `g` is $\nabla f(w)$ with shape `(D,)` and `HV` is `(k, D)` with `HV[j]` $= H\, V_j$. Both float32. Do not build the `(D, D)` Hessian: compute one HVP with `jax.jvp` of `jax.grad(f)` and batch it over the rows of `V` with `jax.vmap`.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def grad_and_hvps(w, X, y, lam, V):
    """Return (gradient (D,), Hessian-vector products (k, D))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 2.0]], "dtype": "float64"}, {"__nd__": [1.0, -1.0], "dtype": "float64"}, 0.0, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [-0.25, 0.5], "dtype": "float32"}, {"__nd__": [[0.125, 0.0], [0.0, 0.5]], "dtype": "float32"}]}}, {"a": [{"__nd__": [0.5, -0.5], "dtype": "float64"}, {"__nd__": [[1.0, 1.0]], "dtype": "float64"}, {"__nd__": [1.0], "dtype": "float64"}, 1.0, {"__nd__": [[1.0, 0.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [0.0, -1.0], "dtype": "float32"}, {"__nd__": [[1.25, 0.25]], "dtype": "float32"}]}}, {"a": [{"__nd__": [-1.165062107645523, -0.33431409226944026, 1.5435583100187176, 0.3472319963906038, 0.19566662474369434, -0.2615339862998367], "dtype": "float64"}, {"__nd__": [[-1.738266398496882, -1.3366427931811324, -1.361106708564987, -0.35161713127840977, -2.3125815796967033, -0.18889719608460778], [-0.957229228096346, 0.8936001849299788, 0.956847237579234, 1.3922582291390866, 0.7674701130947078, -0.053029778757267734], [0.8597939889439096, 1.5054811563838433, -0.6535945334170485, 0.610351145830187, -0.042673827710852374, 1.4400167254152394], [-0.8368950200968434, -0.3015466095655266, 0.36233859316943917, 0.25811026702099754, -1.6394479624476481, 0.360155232237386], [-0.11849769951697288, -0.23974784920156203, -0.15530166200229314, 0.21897170507821917, -1.8163956614546406, 1.5524665679968663], [-0.8614416732885682, -2.2413678581872873, -0.08197449383484104, 1.4574804175244145, -0.5186009711032398, 1.5512756209056682], [1.5569420010285768, -0.8627319171113763, -2.4651208152773547, -1.2351827568078488, 1.1874322225543146, -0.8167721736209068], [-1.5106774902505407, -1.3376946740539473, 0.00018014422029518804, -0.026108483729201715, 0.8720449149097206, 0.9890499704668804], [-0.9322490224957446, -0.15675869871669845, -1.1339346465649662, 0.07189067206172675, -1.1506078219780655, -1.200251555452681], [2.1234219339526375, 0.032388888425520096, 0.6431786123703356, 2.5380983786528484, 0.7858388236905651, -0.11414260193201425], [0.05458995154441833, -0.7444919671299762, 1.094338440936761, -0.3302786854546604, 1.7627455255994957, -0.3447585414426272], [-0.3536229319237997, -0.41937740024787623, 0.6256938797342935, 0.18693606942873356, 1.3714324073778508, -0.5533496226261118], [0.29023814454305935, 1.9379617345617723, -0.24038168695409634, 0.5326127940361377, 1.5718822401651174, 0.5534457799103193], [-0.6345051681060726, -2.062350018165741, 2.3931636444999302, 0.43031428213036543, 1.2605288876070315, -0.4763135491383076], [1.8184817231498083, 0.5195393601930505, 0.04979744331439894, -0.9018171912232378, 0.3681847205816066, -0.48653003243104254], [-0.2227691793707828, -1.6225108173922826, -0.4905539302374794, 0.15987903483797983, -1.250806160003097, 0.07389650956670456], [0.6632938789578444, -0.3067599406971191, -0.6207841599596035, -0.6201241876096135, 0.9411423059808348, -1.5345050452989106], [-0.30045139331057524, -0.4194051074850356, 0.1439970424593328, -0.06957008698046126, -1.1075061566240967, 2.040089487170476], [0.17761176958791589, -1.3420383269747214, -0.891433449836664, 0.0975848755045005, 0.6129699383674597, -1.3421550645818923], [0.7113368741009568, -1.3732920829103488, -0.7943888387036288, 0.8531012625681964, 1.7009972292946562, -0.0847078716217542], [-0.30953861740571575, 1.267899148639513, -1.07197888693, 0.27792183071799254, 0.23324557215707653, 1.0093217602107074], [0.2985784373330526, -0.26639433439756366, 0.6145545843865188, -1.1265648616302042, 0.8140764312836131, -0.6726765092005992], [-0.9103659010549774, -0.36764639009110345, 0.04600144152770579, 0.7105859105367411, -0.21267067374844265, 0.735722851468494], [0.9750824860821719, -0.10575880794402147, -0.981547110530414, 0.9197510892238407, 0.275448247346923, -2.115193535255952], [-0.9882886519638155, 0.1431616902282403, -1.3719055142985772, -0.6814201424601892, -1.7129221407645252, 0.4022636122278133], [0.24151780676325904, 0.17612560137856614, 1.4474740630704988, -0.3526016708078728, -0.28918018544897156, 2.0199232820733553], [-1.3008629733039814, 0.7291629034512065, -1.315096534161614, 0.23625578922278398, 2.1157826250508203, 0.27135602491513133], [-0.7718227720041364, 2.384216059092654, -0.5438213747150857, -0.2061964998368636, 0.3043720602243637, -1.3425651975299016], [1.1150982838459829, 1.618825058923951, 0.7587658147988282, 0.3303610268728072, -0.14929768803644328, 0.8867355822229123], [-3.1500782801905043, -0.12477026219888078, 0.2561257298558171, -1.9855301936747396, 0.42616733808904067, -0.12750498464628504]], "dtype": "float64"}, {"__nd__": [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0], "dtype": "float64"}, 0.1, {"__nd__": [[1.0186587944885543, 0.3292423326847568, -1.0841877220977156, -2.432093302035109, -2.6816922976183717, 0.6576714475598825], [-0.26608447939977387, 1.448129786127416, 0.5099945329233561, -1.2720369952736923, 0.2903889331469363, 1.1918615960129082], [-0.901756687656306, -0.508742366609308, 1.740987177461118, -1.7463920080287247, -1.0902869133462338, 1.050039867134895]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [-0.3152058720588684, -0.2191075086593628, 0.53017258644104, -0.06311485916376114, 0.07636451721191406, -0.05320042744278908], "dtype": "float32"}, {"__nd__": [[-0.08749959617853165, -0.03665420785546303, -0.18834587931632996, -0.5125126242637634, -0.9551136493682861, 0.230600506067276], [-0.053968966007232666, 0.46718138456344604, 0.17509931325912476, -0.25044411420822144, 0.07812806218862534, 0.32688653469085693], [-0.31561458110809326, -0.20673927664756775, 0.32396644353866577, -0.3924764394760132, -0.5399786829948425, 0.43546706438064575]], "dtype": "float32"}]}}, {"a": [{"__nd__": [0.19899573570694631, -1.1424908541942405, -1.4552454113551185, 0.46345475415187043, 1.2142695308238227, 0.8265845125214457], "dtype": "float64"}, {"__nd__": [[-1.738266398496882, -1.3366427931811324, -1.361106708564987, -0.35161713127840977, -2.3125815796967033, -0.18889719608460778], [-0.957229228096346, 0.8936001849299788, 0.956847237579234, 1.3922582291390866, 0.7674701130947078, -0.053029778757267734], [0.8597939889439096, 1.5054811563838433, -0.6535945334170485, 0.610351145830187, -0.042673827710852374, 1.4400167254152394], [-0.8368950200968434, -0.3015466095655266, 0.36233859316943917, 0.25811026702099754, -1.6394479624476481, 0.360155232237386], [-0.11849769951697288, -0.23974784920156203, -0.15530166200229314, 0.21897170507821917, -1.8163956614546406, 1.5524665679968663], [-0.8614416732885682, -2.2413678581872873, -0.08197449383484104, 1.4574804175244145, -0.5186009711032398, 1.5512756209056682], [1.5569420010285768, -0.8627319171113763, -2.4651208152773547, -1.2351827568078488, 1.1874322225543146, -0.8167721736209068], [-1.5106774902505407, -1.3376946740539473, 0.00018014422029518804, -0.026108483729201715, 0.8720449149097206, 0.9890499704668804], [-0.9322490224957446, -0.15675869871669845, -1.1339346465649662, 0.07189067206172675, -1.1506078219780655, -1.200251555452681], [2.1234219339526375, 0.032388888425520096, 0.6431786123703356, 2.5380983786528484, 0.7858388236905651, -0.11414260193201425], [0.05458995154441833, -0.7444919671299762, 1.094338440936761, -0.3302786854546604, 1.7627455255994957, -0.3447585414426272], [-0.3536229319237997, -0.41937740024787623, 0.6256938797342935, 0.18693606942873356, 1.3714324073778508, -0.5533496226261118], [0.29023814454305935, 1.9379617345617723, -0.24038168695409634, 0.5326127940361377, 1.5718822401651174, 0.5534457799103193], [-0.6345051681060726, -2.062350018165741, 2.3931636444999302, 0.43031428213036543, 1.2605288876070315, -0.4763135491383076], [1.8184817231498083, 0.5195393601930505, 0.04979744331439894, -0.9018171912232378, 0.3681847205816066, -0.48653003243104254], [-0.2227691793707828, -1.6225108173922826, -0.4905539302374794, 0.15987903483797983, -1.250806160003097, 0.07389650956670456], [0.6632938789578444, -0.3067599406971191, -0.6207841599596035, -0.6201241876096135, 0.9411423059808348, -1.5345050452989106], [-0.30045139331057524, -0.4194051074850356, 0.1439970424593328, -0.06957008698046126, -1.1075061566240967, 2.040089487170476], [0.17761176958791589, -1.3420383269747214, -0.891433449836664, 0.0975848755045005, 0.6129699383674597, -1.3421550645818923], [0.7113368741009568, -1.3732920829103488, -0.7943888387036288, 0.8531012625681964, 1.7009972292946562, -0.0847078716217542], [-0.30953861740571575, 1.267899148639513, -1.07197888693, 0.27792183071799254, 0.23324557215707653, 1.0093217602107074], [0.2985784373330526, -0.26639433439756366, 0.6145545843865188, -1.1265648616302042, 0.8140764312836131, -0.6726765092005992], [-0.9103659010549774, -0.36764639009110345, 0.04600144152770579, 0.7105859105367411, -0.21267067374844265, 0.735722851468494], [0.9750824860821719, -0.10575880794402147, -0.981547110530414, 0.9197510892238407, 0.275448247346923, -2.115193535255952], [-0.9882886519638155, 0.1431616902282403, -1.3719055142985772, -0.6814201424601892, -1.7129221407645252, 0.4022636122278133], [0.24151780676325904, 0.17612560137856614, 1.4474740630704988, -0.3526016708078728, -0.28918018544897156, 2.0199232820733553], [-1.3008629733039814, 0.7291629034512065, -1.315096534161614, 0.23625578922278398, 2.1157826250508203, 0.27135602491513133], [-0.7718227720041364, 2.384216059092654, -0.5438213747150857, -0.2061964998368636, 0.3043720602243637, -1.3425651975299016], [1.1150982838459829, 1.618825058923951, 0.7587658147988282, 0.3303610268728072, -0.14929768803644328, 0.8867355822229123], [-3.1500782801905043, -0.12477026219888078, 0.2561257298558171, -1.9855301936747396, 0.42616733808904067, -0.12750498464628504]], "dtype": "float64"}, {"__nd__": [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0], "dtype": "float64"}, 0.0, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [0.04726599529385567, -0.21672706305980682, 0.04442821070551872, -0.026346582919359207, 0.17305901646614075, -0.06637930124998093], "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [4.1118532475255725, 14.954732356180166, 3.1114279818219863, -8.324772451972017, 20.318803882508853, -12.34054102590671], "dtype": "float64"}, {"__nd__": [[-1.738266398496882, -1.3366427931811324, -1.361106708564987, -0.35161713127840977, -2.3125815796967033, -0.18889719608460778], [-0.957229228096346, 0.8936001849299788, 0.956847237579234, 1.3922582291390866, 0.7674701130947078, -0.053029778757267734], [0.8597939889439096, 1.5054811563838433, -0.6535945334170485, 0.610351145830187, -0.042673827710852374, 1.4400167254152394], [-0.8368950200968434, -0.3015466095655266, 0.36233859316943917, 0.25811026702099754, -1.6394479624476481, 0.360155232237386], [-0.11849769951697288, -0.23974784920156203, -0.15530166200229314, 0.21897170507821917, -1.8163956614546406, 1.5524665679968663], [-0.8614416732885682, -2.2413678581872873, -0.08197449383484104, 1.4574804175244145, -0.5186009711032398, 1.5512756209056682], [1.5569420010285768, -0.8627319171113763, -2.4651208152773547, -1.2351827568078488, 1.1874322225543146, -0.8167721736209068], [-1.5106774902505407, -1.3376946740539473, 0.00018014422029518804, -0.026108483729201715, 0.8720449149097206, 0.9890499704668804], [-0.9322490224957446, -0.15675869871669845, -1.1339346465649662, 0.07189067206172675, -1.1506078219780655, -1.200251555452681], [2.1234219339526375, 0.032388888425520096, 0.6431786123703356, 2.5380983786528484, 0.7858388236905651, -0.11414260193201425], [0.05458995154441833, -0.7444919671299762, 1.094338440936761, -0.3302786854546604, 1.7627455255994957, -0.3447585414426272], [-0.3536229319237997, -0.41937740024787623, 0.6256938797342935, 0.18693606942873356, 1.3714324073778508, -0.5533496226261118], [0.29023814454305935, 1.9379617345617723, -0.24038168695409634, 0.5326127940361377, 1.5718822401651174, 0.5534457799103193], [-0.6345051681060726, -2.062350018165741, 2.3931636444999302, 0.43031428213036543, 1.2605288876070315, -0.4763135491383076], [1.8184817231498083, 0.5195393601930505, 0.04979744331439894, -0.9018171912232378, 0.3681847205816066, -0.48653003243104254], [-0.2227691793707828, -1.6225108173922826, -0.4905539302374794, 0.15987903483797983, -1.250806160003097, 0.07389650956670456], [0.6632938789578444, -0.3067599406971191, -0.6207841599596035, -0.6201241876096135, 0.9411423059808348, -1.5345050452989106], [-0.30045139331057524, -0.4194051074850356, 0.1439970424593328, -0.06957008698046126, -1.1075061566240967, 2.040089487170476], [0.17761176958791589, -1.3420383269747214, -0.891433449836664, 0.0975848755045005, 0.6129699383674597, -1.3421550645818923], [0.7113368741009568, -1.3732920829103488, -0.7943888387036288, 0.8531012625681964, 1.7009972292946562, -0.0847078716217542], [-0.30953861740571575, 1.267899148639513, -1.07197888693, 0.27792183071799254, 0.23324557215707653, 1.0093217602107074], [0.2985784373330526, -0.26639433439756366, 0.6145545843865188, -1.1265648616302042, 0.8140764312836131, -0.6726765092005992], [-0.9103659010549774, -0.36764639009110345, 0.04600144152770579, 0.7105859105367411, -0.21267067374844265, 0.735722851468494], [0.9750824860821719, -0.10575880794402147, -0.981547110530414, 0.9197510892238407, 0.275448247346923, -2.115193535255952], [-0.9882886519638155, 0.1431616902282403, -1.3719055142985772, -0.6814201424601892, -1.7129221407645252, 0.4022636122278133], [0.24151780676325904, 0.17612560137856614, 1.4474740630704988, -0.3526016708078728, -0.28918018544897156, 2.0199232820733553], [-1.3008629733039814, 0.7291629034512065, -1.315096534161614, 0.23625578922278398, 2.1157826250508203, 0.27135602491513133], [-0.7718227720041364, 2.384216059092654, -0.5438213747150857, -0.2061964998368636, 0.3043720602243637, -1.3425651975299016], [1.1150982838459829, 1.618825058923951, 0.7587658147988282, 0.3303610268728072, -0.14929768803644328, 0.8867355822229123], [-3.1500782801905043, -0.12477026219888078, 0.2561257298558171, -1.9855301936747396, 0.42616733808904067, -0.12750498464628504]], "dtype": "float64"}, {"__nd__": [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0], "dtype": "float64"}, 0.01, {"__nd__": [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [0.2176029086112976, 0.26071518659591675, 0.20020164549350739, -0.1930592954158783, 0.6782615780830383, -0.4568603038787842], "dtype": "float32"}, {"__nd__": [[0.017675859853625298, 0.015283867716789246, -0.010142537765204906, 0.003322803881019354, -0.0032912343740463257, 0.01107319351285696], [0.015283867716789246, 0.045323409140110016, -0.02708168514072895, 0.004285997245460749, -0.010257665067911148, 0.02205660380423069], [-0.010142538696527481, -0.0270816832780838, 0.035601817071437836, 0.0005146656767465174, 0.01143063884228468, -0.012339238077402115], [0.0033228048123419285, 0.004285996779799461, 0.0005146663170307875, 0.013896433636546135, 0.0019099388737231493, 0.006498614326119423], [-0.0032912346068769693, -0.010257666930556297, 0.011430641636252403, 0.0019099394557997584, 0.016078881919384003, -0.002723515499383211], [0.011073190718889236, 0.022056609392166138, -0.012339235283434391, 0.006498614326119423, -0.002723515499383211, 0.028332393616437912]], "dtype": "float32"}]}}, {"a": [{"__nd__": [0.0, 0.0, 0.0, 0.0], "dtype": "float64"}, {"__nd__": [[-0.1345385182665012, -0.07306731612303265, 1.4311362879789098, -0.76370324018215], [-0.27672012197919366, -1.7804064168339873, 0.3375348043583882, 0.44516248734272634], [0.650677300056639, -0.9267338354218425, 0.2363626046492526, -0.8841448459240736], [-0.4791188612131528, -0.07857993937359609, 1.2926464672619307, -0.4402814068158908], [0.17397289568303775, -0.8474342449998818, -0.5732844194612009, -0.7051883438851831], [-0.16304818564259638, 0.7618706015488411, -1.1113026843689302, 0.7127338161258466], [-0.9589356443816781, 0.32700751395201577, -0.560870965666268, -0.31224180271226776], [-0.4021692347232762, 1.0870613289026454, 0.2394661460452593, 1.0559872773133139], [-0.38340030439777556, -0.2582603578332374, 1.5923687082932494, -1.6704613982986534], [-0.9014681893014775, -0.2014796354504079, 1.7977227001980955, -0.633399520432185], [0.7699411198902094, 1.5583218731168709, -0.2608745394559887, -0.19837509448756974], [1.6461708345876043, -0.5787852279754228, -1.0786494930343222, 0.8029630260193307]], "dtype": "float64"}, {"__nd__": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], "dtype": "float64"}, 2.0, {"__nd__": [[-0.5418881870963893, 1.2721842522288556, 0.5135474956511674, -1.7799098873321286], [0.25332469913849437, -0.014213846692776116, -0.6893300001570001, 1.0785818356114014], [-1.797825175632008, -1.2800208576174648, -0.5234266949164023, 0.7662141376599587], [-1.5694801544923136, -0.6661272213169701, -0.7451257621496428, -1.5424983132721142], [-0.41986233875211537, -1.18571085332387, -0.4889876495432731, 0.2486882945988942]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [0.01910986751317978, 0.04210356995463371, -0.1392606496810913, 0.10795622318983078], "dtype": "float32"}, {"__nd__": [[-1.2759249210357666, 2.704982280731201, 1.3982996940612793, -3.8813250064849854], [0.6396308541297913, 0.03954702243208885, -1.7270981073379517, 2.431945323944092], [-3.7366371154785156, -2.738755464553833, -1.0712467432022095, 1.6095640659332275], [-3.3158175945281982, -1.4839946031570435, -1.3161258697509766, -3.321915626525879], [-0.8260844349861145, -2.5713400840759277, -1.0563266277313232, 0.536296010017395]], "dtype": "float32"}]}}, {"a": [{"__nd__": [-0.9534118342920758, -0.9142826632962725, -1.0316511638293742], "dtype": "float64"}, {"__nd__": [[-0.8933054947581841, -1.6148024657827056, -1.2263999142508761], [0.07719191957678519, -0.9243544583504244, 3.2339018997770634], [-0.8600130621304797, 0.4160232040597361, -1.5521503681442799], [-0.31256815258660375, -0.5912597525566127, -0.12163392660830079], [1.2945406371939017, -0.20827333894827663, -1.2501996254287515], [0.08347004433274519, -1.0038149259644478, -0.15295570559470914], [1.5730545295598348, 0.9302760443668044, 1.0671787309950316], [0.5212999120526557, -0.031584323505555144, -1.55792171748771], [-0.4846561933752778, -0.5406911118615139, 0.6760981800274841], [0.9945112303009566, 0.4785252373086558, -0.7995845410759872], [1.8504661621271392, 1.4198445308034204, -0.217647563808852], [0.19711457610825392, 0.6402798930741478, 0.1038913332108938], [-0.0985355893800558, 0.49441975839418695, 0.010406093909436507], [0.20431136185333393, -1.61135015473308, -1.6075892924212571], [1.7410867124614156, -1.522369386938202, 0.48722882717261135], [0.3203769693891191, 0.22250875841787351, -0.6936047282095571], [1.0157777010524436, 0.3510181349783536, -0.34216112927079706], [-1.4072133498853296, -0.22970756507211673, -0.598956468536741], [-1.2583181039046392, 0.38112053300236753, 0.5828281330045602], [0.8354563243155745, 0.12421026932529662, -0.2383621125961844], [1.1490908658564791, 0.7832782475596382, 2.32074907968081], [0.5771483062139039, -0.9177967306386784, 0.9884631944811776], [2.3570267318099143, 0.29937769005829046, -0.15610041187032964], [-0.5687140108804395, 1.9433252799360246, -0.15373294449854558], [-0.2112595156569885, 0.34181049088817905, -1.3857484195269552], [-0.23304622770269062, -1.2283991582206826, -0.09942019984254838], [0.54260422644422, 0.9742494560082414, -0.37179205318903635], [0.8710718051633862, -0.34345632027391115, -1.6070256610252387], [0.6258801985153036, -0.15362507105786222, 0.16474390375755757], [-2.5691738029948086, 0.2735448032045201, 0.31109184486697117], [0.31143821385495585, 1.10896493931132, -0.29332033465087887], [2.2237239826396973, 0.8031034492668545, 0.6240062021845986], [-2.4214827638407037, -1.8931089335359528, 1.5844553249260978], [0.4764431787137526, 1.020111628826446, -0.29070837842375824], [-0.027039805602453622, 0.6851477742540288, -0.6922662409590742], [1.4292187639176546, -1.0044193742809302, 0.33179445579984257], [-0.15690743804118637, 0.47324409918150234, 1.3712556497520516], [-0.9960409165773517, 0.584417712265637, 0.2546463632537991], [0.9801493919813679, 0.11495698172851479, -0.3519676422063378], [0.6444337404067579, 1.4782007160392028, 0.9628000007386113], [0.3031577187509966, 0.5738712367020865, -1.393775189850605], [-0.3917389526632646, -0.10254844496126256, -0.3870595211594539], [-0.5505884955391186, 0.10272799001211093, 0.5513552694553776], [2.090562209365477, -0.5512598430291946, -0.7597450770639842], [1.1461411757295485, 0.30457401615876917, -0.7397732608579033], [1.6805783917225179, -0.06008251955141739, -0.17854256124526102], [-0.4675795298506181, -0.7756847304742605, 0.2452337387678607], [-0.8146144933956802, 2.5966355916351644, 0.4721570691520021], [-0.11234265778539619, -0.6410627914420706, 0.3851921986841414], [1.1213703471446088, 0.3636777921686434, -1.0227874595910413]], "dtype": "float64"}, {"__nd__": [1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0], "dtype": "float64"}, 0.05, {"__nd__": [[-0.13237423930322, 0.22949164643491693, -0.17156587390725997]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [-0.29187262058258057, -0.2886839509010315, -0.1982712596654892], "dtype": "float32"}, {"__nd__": [[-0.0231358353048563, 0.04084168002009392, -0.026489656418561935]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(grad_and_hvps, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")